In [1]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 721.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.1/985.1 kB 13.3 MB/s eta 0:00:00


In [2]:
!pip install scikit-learn==1.2.2
import sklearn
sklearn.__version__

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 34.4 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.5.2
    Uninstalling scikit-learn-1.5.2:
      Successfully uninstalled scikit-learn-1.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlxtend 0.23.3 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.


'1.2.2'

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy
from sklearn.metrics import davies_bouldin_score
from collections import Counter


In [4]:
pypfopt.__version__

'1.5.5'

In [5]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.50
    Uninstalling yfinance-0.2.50:
      Successfully uninstalled yfinance-0.2.50
yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [6]:
def modify_date(date):
    return date[:10]

In [7]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [8]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [9]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [10]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [11]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [12]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

# Step 1 - Preselection

In [13]:
# Preselection parameters set up
#method = "K_means_multiple"# or avg or anyother method
method = "K_means_multiple"
stock_type = "S&P100"
end_swan = "Swan8"         #equals the last swan before the swan we are going to predict
start_swan="Swan4"
start = dt(2019,1,1)
end = dt(2020,2,15)
selected_stock_number = 40     #number of stock preselected
start_swan_date = dt(2020,2,15)
end_swan_date = dt(2021,3,23)
w1 = 0.1
w2 = 0.1
MA_days = 30
N = 6 # This parameter is set to help classification method like K-means
Min_appear = 1
Seeds = 40

In [14]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
    #portfolio_beta = pd.read_excel("BetaMatrix S&P100 Cathy(Yijue) 2022.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("SP500_BETALouise 8_30_24 (2).xlsx",index_col=0)

optdata = pd.read_excel("optdata.xlsx",index_col=0,sheet_name = 1)
portfolio_beta = portfolio_beta[portfolio_beta.index.isin(optdata.index)]

In [15]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [16]:
portfolio_beta

,Swan -4,Swan -3,Swan -2,Swan -1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,,,,,
SO,NaN,NaN,NaN,0.789771,0.735961,-0.130115,-0.265152,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1.173178,0.264469
DUK,NaN,NaN,-0.076048,0.180670,0.400895,-0.126448,-0.519440,1.580780,0.771715,-0.322795,0.233695,-0.356252,1.165454,0.534699
NEE,0.982309,0.253298,-0.134131,0.609003,0.579708,0.166846,-0.420656,0.414536,0.503895,0.415370,0.192031,-0.026634,1.062981,0.710514
VZ,NaN,NaN,NaN,-0.071253,0.547390,0.059871,0.109521,0.905674,0.332333,0.064046,0.717548,0.060183,0.430888,1.036945
EXC,1.101894,0.665427,-0.825250,1.115449,0.634396,0.174697,-0.975565,0.519666,0.794675,0.263392,0.605247,-0.123437,1.254518,0.353268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
JPM,NaN,NaN,0.873322,1.666279,1.196511,2.138915,1.576224,1.649018,1.453522,1.887942,1.012903,0.969234,1.348489,1.245036
BAC,2.119191,1.270985,0.770965,1.317948,0.992028,2.385911,1.731738,0.594107,2.142401,2.667371,1.184258,1.192922,1.509426,1.256673
MS,NaN,NaN,NaN,NaN,NaN,NaN,2.109127,1.441339,0.770028,2.806836,1.528749,1.047092,1.605162,0.793801


In [17]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [18]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8
Swan1,1.0,-0.000398,0.080647,0.060068,-0.000300,0.074837,0.084818,0.059016
Swan2,NaN,1.000000,0.597924,0.362586,0.433151,0.448483,0.040243,0.368497
Swan3,NaN,NaN,1.000000,0.162676,0.291646,0.407865,0.222270,0.191530
Swan4,NaN,NaN,NaN,1.000000,-0.012590,0.174132,0.117535,0.239998
Swan5,NaN,NaN,NaN,NaN,1.000000,0.455397,0.292794,0.203132
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.351467,0.455204
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.180964
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [19]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [20]:
portfolio_beta

,Swan -4,Swan -3,Swan -2,Swan -1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,,,,,
SO,NaN,NaN,NaN,0.789771,0.735961,-0.130115,-0.265152,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1.173178,0.264469
DUK,NaN,NaN,-0.076048,0.180670,0.400895,-0.126448,-0.519440,1.580780,0.771715,-0.322795,0.233695,-0.356252,1.165454,0.534699
NEE,0.982309,0.253298,-0.134131,0.609003,0.579708,0.166846,-0.420656,0.414536,0.503895,0.415370,0.192031,-0.026634,1.062981,0.710514
VZ,NaN,NaN,NaN,-0.071253,0.547390,0.059871,0.109521,0.905674,0.332333,0.064046,0.717548,0.060183,0.430888,1.036945
EXC,1.101894,0.665427,-0.825250,1.115449,0.634396,0.174697,-0.975565,0.519666,0.794675,0.263392,0.605247,-0.123437,1.254518,0.353268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
JPM,NaN,NaN,0.873322,1.666279,1.196511,2.138915,1.576224,1.649018,1.453522,1.887942,1.012903,0.969234,1.348489,1.245036
BAC,2.119191,1.270985,0.770965,1.317948,0.992028,2.385911,1.731738,0.594107,2.142401,2.667371,1.184258,1.192922,1.509426,1.256673
MS,NaN,NaN,NaN,NaN,NaN,NaN,2.109127,1.441339,0.770028,2.806836,1.528749,1.047092,1.605162,0.793801


In [21]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,Seeds):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    stock_frequency_name = f"Astock_frequency_w1_{w1}_Seeds_{Seeds}.csv"
    df.to_csv(stock_frequency_name)

    for p in range(0,min(selected_stock_number,len(sorted_count))):
      if sorted_count[p][1]>=Min_appear:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

elif method == "Custom":
    # Step 1: Calculate Davies-Bouldin index for random seeds
    selected_beta = portfolio_beta.loc[:, start_swan:end_swan].dropna()
    dbi_scores = []

    # Loop through random seeds from 1 to 1000 to find the best seeds with lowest DBI
    for R in range(1, 1001):
        kmeans = KMeans(n_clusters=N, random_state=R)
        labels = kmeans.fit_predict(selected_beta)
        dbi = davies_bouldin_score(selected_beta, labels)
        dbi_scores.append((R, dbi))

    # Sort by Davies-Bouldin index and select the top 5 seeds with lowest DBI
    top_seeds = sorted(dbi_scores, key=lambda x: x[1])[:Seeds]


    stock_counter = Counter()

    for R, _ in top_seeds:

        selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
        Y_predic = selected_beta[end_swan].copy()
        Kmean = KMeans(n_clusters=N,random_state=R)#set the random_state to keep modeling the same every run
        Kmean.fit(selected_beta)
        ct = Kmean.cluster_centers_
        Kmean.labels_
        selected_beta["Group"]=Kmean.labels_
        res = ct.mean(axis=1)
        res [res<0]=0

        min_group = np.where(res==min(res))
        for i in range(N):
            Y_predic[selected_beta["Group"] == i] = res [i]
        selected_beta ["avg"]=Y_predic.values
        selected_beta=selected_beta.sort_values(by=["avg"])
        first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
        if first_group.shape[0] <10:
            second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
            preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=True)
        else:
            preselected_stock = first_group

        stock_counter.update(preselected_stock.index)

    top_stocks = [stock for stock, count in stock_counter.most_common(22)]

<ipython-input-21-245ea1449c5f>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-21-245ea1449c5f>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-21-245ea1449c5f>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_b

['SO', 'AMGN', 'MDLZ', 'MRK', 'JNJ', 'CL', 'PEP', 'LLY', 'SBUX', 'KO', 'PG', 'VZ', 'EXC', 'NEE', 'WMT', 'ABT', 'SO', 'SBUX', 'MDLZ', 'MRK', 'JNJ', 'CL', 'PEP', 'LLY', 'ABT', 'KO', 'PG', 'VZ', 'EXC', 'NEE', 'WMT', 'SO', 'AMGN', 'SBUX', 'MDLZ', 'MRK', 'JNJ', 'ABT', 'PEP', 'LLY', 'CL', 'KO', 'NEE', 'PG', 'WMT', 'VZ', 'EXC', 'SO', 'ABT', 'AMGN', 'SBUX', 'PFE', 'MDLZ', 'MRK', 'JNJ', 'PEP', 'CL', 'PG', 'NEE', 'VZ', 'KO', 'EXC', 'LLY', 'WMT', 'SO', 'COST', 'AMZN', 'ABT', 'AMGN', 'PFE', 'MDLZ', 'MRK', 'JNJ', 'CL', 'SBUX', 'LLY', 'PG', 'NEE', 'KO', 'MO', 'WMT', 'EXC', 'PEP', 'VZ', 'SO', 'COST', 'AMZN', 'ABT', 'AMGN', 'PFE', 'MDLZ', 'MRK', 'JNJ', 'CL', 'SBUX', 'LLY', 'PG', 'NEE', 'KO', 'MO', 'WMT', 'EXC', 'PEP', 'VZ', 'SO', 'ABT', 'AMGN', 'SBUX', 'MDLZ', 'MRK', 'JNJ', 'PEP', 'LLY', 'CL', 'KO', 'NEE', 'MO', 'VZ', 'WMT', 'EXC', 'PG', 'SO', 'COST', 'AMZN', 'ABT', 'AMGN', 'PFE', 'MDLZ', 'MRK', 'JNJ', 'CL', 'SBUX', 'WMT', 'NEE', 'LLY', 'PG', 'KO', 'MO', 'PEP', 'EXC', 'VZ', 'SO', 'COST', 'ABT', 'SBUX'

<ipython-input-21-245ea1449c5f>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-21-245ea1449c5f>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-21-245ea1449c5f>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_b

In [22]:
ct

array([[ 2.54565631,  0.63571697,  1.25106214, -0.25339946,  2.78743065],
       [ 1.32065521,  0.66078065,  0.54363846,  0.85368854,  0.9562835 ],
       [ 1.20159506,  1.23587438,  1.57510315,  1.19122769,  1.23141796],
       [ 0.5272343 ,  0.66440289,  0.25129497,  0.65051639,  0.10229729],
       [ 0.26114982,  1.14374197,  0.95053372,  0.63134428,  0.91336698],
       [ 0.87027788,  2.08299487,  2.2685296 ,  1.24180872,  1.25974987]])

In [23]:
res

array([1.39329332, 0.86700927, 1.28704365, 0.43914917, 0.78002736,
       1.54467219])

In [24]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22 entries, ABT to WMT
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Swan4   22 non-null     float64
 1   Swan5   22 non-null     float64
 2   Swan6   22 non-null     float64
 3   Swan7   22 non-null     float64
 4   Swan8   22 non-null     float64
 5   Group   22 non-null     int32  
 6   avg     22 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 1.3+ KB


In [25]:
#order = list(preselected_stock.index)
#random.shuffle(order)
#preselected_stock = preselected_stock.reindex(index = order)
preselected_stock


,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg
Ticker,,,,,,,
ABT,0.702187,0.474695,0.311080,1.073151,0.431920,3,0.598607
AMGN,0.795356,0.502892,0.109295,0.843475,0.659977,3,0.582199
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,4,0.380388
CL,0.099786,0.617541,-0.045655,0.784583,0.569564,3,0.405164
COST,0.829819,0.976293,-0.015789,0.618965,0.918644,1,0.665586
DUK,1.580780,0.771715,-0.322795,0.233695,-0.356252,3,0.381429
EXC,0.519666,0.794675,0.263392,0.605247,-0.123437,3,0.411909
JNJ,0.403159,0.739962,0.360347,0.668745,0.555245,3,0.545492
KO,-0.170561,0.536675,0.325968,0.614677,-0.049527,3,0.251446


In [26]:
#holt step
portfolio_beta = pd.read_csv("HoltestimateS9.csv",index_col=0)
if 'Ticker' in preselected_stock.columns:
    preselected_stock.set_index('Ticker', inplace=True)
preselected_stock['avg'] = preselected_stock.index.map(portfolio_beta['Forecasted Swan9'])

In [27]:
#using average step
'''
preselected_stock

preselected_stock['avg'] = (preselected_stock['Swan4'] + preselected_stock['Swan5']+preselected_stock['Swan6']+preselected_stock['Swan7']+preselected_stock['Swan8'])/5
'''

"\npreselected_stock\n\npreselected_stock['avg'] = (preselected_stock['Swan4'] + preselected_stock['Swan5']+preselected_stock['Swan6']+preselected_stock['Swan7']+preselected_stock['Swan8'])/5\n"

In [28]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.to_csv('preselected_stocks.csv')
preselected_stock

,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg,PR
Ticker,,,,,,,,
ABT,0.702187,0.474695,0.311080,1.073151,0.431920,3,0.615856,0.384144
AMGN,0.795356,0.502892,0.109295,0.843475,0.659977,3,0.603026,0.396974
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,4,1.586709,-0.586709
CL,0.099786,0.617541,-0.045655,0.784583,0.569564,3,0.737313,0.262687
COST,0.829819,0.976293,-0.015789,0.618965,0.918644,1,0.611810,0.388190
DUK,1.580780,0.771715,-0.322795,0.233695,-0.356252,3,-0.942127,1.942127
EXC,0.519666,0.794675,0.263392,0.605247,-0.123437,3,-0.030782,1.030782
JNJ,0.403159,0.739962,0.360347,0.668745,0.555245,3,0.615227,0.384773
KO,-0.170561,0.536675,0.325968,0.614677,-0.049527,3,0.347470,0.652530


# Step 2 - Download data and adjuste price based on Swan Beta

In [29]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [30]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['ABT' 'AMGN' 'AMZN' 'CL' 'COST' 'DUK' 'EXC' 'JNJ' 'KO' 'LLY' 'MCD' 'MDLZ'
 'MO' 'MRK' 'NEE' 'PEP' 'PFE' 'PG' 'SBUX' 'SO' 'VZ' 'WMT']


In [31]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]

[*********************100%%**********************]  22 of 22 completed


In [32]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame=yahooDataFrame[selected_stock_list]
yahooDataFrame

Ticker,ABT,AMGN,AMZN,CL,COST,DUK,EXC,JNJ,KO,LLY,...,MO,MRK,NEE,PEP,PFE,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,62.657761,159.867584,76.956497,51.385075,186.350327,66.240883,25.484903,108.155205,38.957546,105.295586,...,31.274710,60.730328,36.857189,91.920586,31.940189,78.474571,56.738701,34.275475,40.790035,28.355379
2019-01-03,59.700668,157.435028,75.014000,51.142231,182.400497,66.217430,25.513819,106.436569,38.716812,102.023422,...,31.103472,59.485020,36.766041,91.062630,31.046593,77.924370,54.277561,34.761562,40.935650,28.209564
2019-01-04,61.404602,162.816727,78.769501,51.619236,187.697235,66.764900,25.802961,108.222931,39.488823,105.093925,...,31.902618,61.276649,37.443165,92.929955,31.755558,79.514832,56.077103,35.051624,41.037594,28.385757
2019-01-07,62.324196,165.007690,81.475502,51.931446,188.388870,66.483337,25.849224,107.528709,38.974136,105.662201,...,30.938559,60.601757,37.388901,92.130867,31.925415,79.196739,56.077103,34.973232,41.299717,28.719921
2019-01-08,61.575905,167.123718,82.829002,52.512508,189.799561,67.320145,26.248238,110.026199,39.414104,106.633774,...,31.116154,61.051697,37.662350,93.014076,32.073120,79.489044,56.121212,35.968880,42.508423,28.920422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,81.270958,195.220032,106.695503,69.203598,288.990509,78.532730,28.818161,132.191315,51.171310,136.381073,...,31.278984,70.695503,60.400379,126.156670,29.255291,112.183098,79.029457,56.103107,46.195118,35.720390
2020-02-11,82.071678,196.541977,107.540001,68.094215,286.126251,79.152962,29.432589,132.304504,51.034286,135.200134,...,30.852446,70.357132,60.491501,126.520432,29.471939,110.395927,79.011375,56.226059,45.006489,35.766891
2020-02-12,81.721947,194.859497,108.000000,67.951042,290.215363,79.911911,29.384867,131.521042,50.880123,134.506561,...,30.737354,68.706520,60.071423,126.520432,29.201134,109.764626,79.968452,56.250660,44.653740,35.906361


In [33]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [34]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [35]:
market_drop.to_csv("market_drop.csv")

In [36]:
market_drop

,0
0,0.0000
1,0.0037
2,0.0074
3,0.0111
4,0.0148
...,...
278,0.0000
279,0.0000
280,0.0000
281,0.0000


In [37]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [38]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['ABT', 'AMGN', 'AMZN', 'CL', 'COST', 'DUK', 'EXC', 'JNJ', 'KO',
       'LLY', 'MCD', 'MDLZ', 'MO', 'MRK', 'NEE', 'PEP', 'PFE', 'PG',
       'SBUX', 'SO', 'VZ', 'WMT'], dtype=object)

In [39]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame

Ticker,ABT,AMGN,AMZN,CL,COST,DUK,EXC,JNJ,KO,LLY,...,MO,MRK,NEE,PEP,PFE,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,62.657761,159.867584,76.956497,51.385075,186.350327,66.240883,25.484903,108.155205,38.957546,105.295586,...,31.274710,60.730328,36.857189,91.920586,31.940189,78.474571,56.738701,34.275475,40.790035,28.355379
2019-01-03,59.564630,157.083759,74.573606,51.002712,181.987598,66.448255,25.516725,106.194283,38.667036,101.983812,...,30.896778,59.420178,36.799411,91.025208,30.989411,77.839376,54.310797,34.776067,40.931980,28.148058
2019-01-04,61.124761,162.090174,77.844617,51.337595,186.847456,67.230368,25.808839,107.730226,39.387286,105.012321,...,31.478609,61.143061,37.511134,92.853577,31.638583,79.341374,56.145779,35.080875,41.030236,28.261977
2019-01-07,61.898147,163.903196,80.040517,51.506430,187.109504,67.178594,25.858056,106.794394,38.823816,105.539132,...,30.321764,60.403581,37.490706,92.017285,31.749014,78.937593,56.180117,35.017010,41.288610,28.532065
2019-01-08,61.014661,165.632174,80.883904,51.939479,188.080966,68.258821,26.260196,109.024371,39.211415,106.468173,...,30.289040,60.785500,37.799083,92.861182,31.836831,79.142241,56.258672,36.028912,42.493180,28.668198
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,81.270958,195.220032,106.695503,69.203598,288.990509,78.532730,28.818161,132.191315,51.171310,136.381073,...,31.278984,70.695503,60.400379,126.156670,29.255291,112.183098,79.029457,56.103107,46.195118,35.720390
2020-02-11,82.071678,196.541977,107.540001,68.094215,286.126251,79.152962,29.432589,132.304504,51.034286,135.200134,...,30.852446,70.357132,60.491501,126.520432,29.471939,110.395927,79.011375,56.226059,45.006489,35.766891
2020-02-12,81.721947,194.859497,108.000000,67.951042,290.215363,79.911911,29.384867,131.521042,50.880123,134.506561,...,30.737354,68.706520,60.071423,126.520432,29.201134,109.764626,79.968452,56.250660,44.653740,35.906361


# Step3 - MVO

In [40]:
#Set the paramter for MVO process
Portfolio_Date=end

In [41]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [42]:
#calculated rf
#bond_yield = pd.read_csv("bond_yield.csv", index_col=0)
#bond_yield=np.log(bond_yield/100+1)
#date=list(map(str,bond_yield.index.values))
#date=list(map(modify_date,date))
#week=pd.to_datetime(date).strftime("%W").values
#week_1=np.roll(week,1)
#week_difference=np.where(week!=week_1)
#week_difference=np.delete(week_difference,0,None)
#rf_week=bond_yield.iloc[week_difference-1]

In [43]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)

In [44]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [45]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [46]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [47]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [48]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [49]:
# Read OPT-data
optdata = pd.read_excel("optdata.xlsx",index_col=0,sheet_name = 1)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [50]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [51]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)

/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [52]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [53]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)

OrderedDict([('ABT', 0.0), ('AMGN', 0.0), ('AMZN', 0.1), ('CL', 0.0), ('COST', 0.09811), ('DUK', 0.1), ('EXC', 0.0), ('JNJ', 0.0), ('KO', 0.04679), ('LLY', 0.1), ('MCD', 0.1), ('MDLZ', 0.0), ('MO', 0.00785), ('MRK', 0.05046), ('NEE', 0.1), ('PEP', 0.02601), ('PFE', 0.00589), ('PG', 0.06489), ('SBUX', 0.0), ('SO', 0.1), ('VZ', 0.0), ('WMT', 0.1)])
Expected annual return: 17.1%
Annual volatility: 7.4%
Sharpe Ratio: 2.19


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.17136947816747272, 0.07373198610897738, 2.1885952987752875)

In [54]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [55]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 2.188595298240311


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [56]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.030]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [57]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  11 of 11 completed


In [58]:
stock_price.to_csv("tradeinfo.csv")
stock_price=stock_price[weight.index]
stock_price

Ticker,AMZN,COST,DUK,KO,LLY,MCD,MRK,NEE,PG,SO,WMT
Date,,,,,,,,,,,
2020-02-18,107.783501,296.597748,84.069229,50.982899,132.848404,192.522049,68.054527,62.689774,111.027206,57.401844,37.077938
2020-02-19,108.511002,297.491058,83.574799,51.188442,132.678848,192.058929,67.674881,61.051655,111.534012,56.748417,36.473549
2020-02-20,107.654999,298.467316,83.583054,51.145626,133.781113,191.569031,68.079300,61.576214,112.547646,57.740952,36.476650
2020-02-21,104.798500,296.505585,84.407089,51.496758,133.818787,192.272659,67.955498,61.700672,112.654343,57.062714,36.752495
2020-02-24,100.464500,288.833893,84.299965,50.229252,130.153931,190.179581,67.121941,60.889393,109.631256,56.078445,36.052032
...,...,...,...,...,...,...,...,...,...,...,...
2021-03-16,154.593002,311.866119,80.055679,45.744759,183.273483,201.947159,65.978973,68.760597,117.488159,52.944164,42.032833
2021-03-17,156.786499,313.714966,79.048737,45.762619,179.270065,205.850891,66.399879,66.553108,117.014343,52.468739,41.683056
2021-03-18,151.399506,307.796875,79.461845,45.164230,176.581940,204.739456,66.374115,64.883820,117.269463,52.209423,41.138802


In [59]:
weight

,weight
AMZN,0.104140
COST,0.102171
DUK,0.104140
KO,0.048727
LLY,0.104140
MCD,0.104140
MRK,0.052549
NEE,0.104140
PG,0.067576
SO,0.104140


In [60]:
start=stock_price.loc["2020-02-19"]
end = stock_price.loc["2020-03-23"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2020-03-23"]/OEX.loc["2020-02-19"]-1)

In [61]:
beta = stock_drop/market_drop
1-beta.T@weight

,0
weight,0.254731


In [62]:
beta

,0
Ticker,
AMZN,0.390752
COST,0.368082
DUK,1.165454
KO,1.161008
LLY,0.490481
MCD,1.142392
MRK,0.582135
NEE,1.062981
PG,0.701342


In [63]:
stock_drop

,0
Ticker,
AMZN,0.123209
COST,0.116061
DUK,0.367482
KO,0.366080
LLY,0.154655
MCD,0.360210
MRK,0.183554
NEE,0.335171
PG,0.221142


In [64]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.17:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.17)
    # row['weight']=0.17

for i,row in weight.iterrows():
  if row['weight']<0.165:
    # row['weight']=row['weight']+row['weight']/rest*extra
    a=1


weights_name = f"Aweights_w1_{w1}_Seeds_{Seeds}.csv"
weight

,weight
AMZN,0.104140
COST,0.102171
DUK,0.104140
KO,0.048727
LLY,0.104140
MCD,0.104140
MRK,0.052549
NEE,0.104140
PG,0.067576
SO,0.104140


In [65]:
portfolio_beta = portfolio_beta.loc[weight.index]

swan_beta_name = f"A_forcast_and_actual_Beta{w1}_Seeds_{Seeds}.csv"

portfolio_beta

,Forecasted Swan9,Actual Swan9
AMZN,1.586709,0.390752
COST,0.611810,0.368083
DUK,-0.942127,1.165454
KO,0.347470,1.161009
LLY,0.104932,0.490481
MCD,-0.431662,1.142392
MRK,0.294607,0.582136
NEE,-0.245304,1.062981
PG,0.294791,0.701342
SO,-0.112772,1.173178


In [66]:
a=stock_price@weight
b=1-a.loc["2020-03-23"]/a.loc["2020-02-19"]
1-b/market_drop

,0
weight,0.300847


In [67]:
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T


In [68]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX[0])
injected_result["oex"]=S2*OEX

<ipython-input-68-1f2d32375bd9>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])


In [69]:
temp=injected_result.loc[["2020-02-19","2020-03-23"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print(PR)
temp

0.25554655405283166


<ipython-input-69-0cde1dbf7bf2>:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-69-0cde1dbf7bf2>:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


,portfolio_value,oex
Date,,
2020-02-19,9946.222901,10051.749340
2020-03-23,7611.495465,6882.312479


In [70]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


<ipython-input-11-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<ipython-input-11-2b9c13413cb0>:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfMVOweight.iloc[i,5]=allocationweight[0]
<ipython-input-11-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<i

,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
NEE,0.104140,5.403827,18.830078,0.090909,0.581048,0.11,0.11
DUK,0.104140,2.190179,1.271757,0.090909,0.235499,0.11,0.11
WMT,0.104140,1.459465,12.626614,0.090909,0.156929,0.11,0.11
SO,0.104140,1.45499,1.709654,0.090909,0.156448,0.11,0.11
LLY,0.104140,1.068322,1.424596,0.090909,0.114871,0.11,0.11
MRK,0.052549,0.840457,1.41961,0.090909,0.09037,0.11,0.11
PG,0.067576,0.631906,0.802916,0.090909,0.067946,0.11,0.11
KO,0.048727,0.564741,0.749026,0.090909,0.060724,0.11,0.0575
COST,0.102171,0.376396,0.940158,0.090909,0.040472,0.11,0.0575
MCD,0.104140,0.0,0.0,0.090909,0.0,0.11,0.0575


In [71]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  11 of 11 completed
<ipython-input-12-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  11 of 11 completed
<ipython-input-12-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  11 of 11 

#

In [72]:
Portfolio_DD_L = abs(injected_result['portfolio_value'].iloc[24] / injected_result['portfolio_value'].iloc[1] - 1)
DD_L = abs(injected_result['oex'].iloc[24] / injected_result['oex'].iloc[1] - 1)
PR_L = 1 - (Portfolio_DD_L / DD_L)

print(f"Portfolio Drawdown: {Portfolio_DD_L}")
print(f"DD: {DD_L}")
print(f"Performance Ratio (PR): {PR_L}")

Portfolio Drawdown: 0.23473508073763705
DD: 0.3153119674783472
Performance Ratio (PR): 0.25554655405283166


In [73]:
injected_result_name = f"AInjected_result_w1_{w1}_Seeds_{Seeds}.csv"
injected_result.to_csv(injected_result_name)

In [74]:
gradient_matrix = weight.join(portfolio_beta, how='outer')
gradient_matrix_name = f"AAAgradient_matrix_Weight:_{w1}_Seeds:_{Seeds}.csv"

In [75]:
file_name = f"AAA_gradient_matrix_Weight:_{w1}_Seeds:_{Seeds}.xlsx"

with pd.ExcelWriter(file_name) as writer:
    gradient_matrix.to_excel(writer, sheet_name='Sheet1 - Weights and Beta')
    injected_result.to_excel(writer, sheet_name='Sheet2 - injected_result')
    df.to_excel(writer, sheet_name='Sheet3 - frequency')